In [45]:
import numpy as np
import pandas as pd
import nltk
import string
import re

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\matthieu.catteyfaye\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [46]:
df = pd.read_csv("../data/SMS_test.csv", encoding='latin1')

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   S. No.        125 non-null    int64
 1   Message_body  125 non-null    str  
 2   Label         125 non-null    str  
dtypes: int64(1), str(2)
memory usage: 3.1 KB


a. Text Normalization (Lowercasing, Handling contractions, Spell schecking)

In [48]:
# Lowercasing

df["Case Normalization"] = df["Message_body"].str.lower()
df.head(1)

,S. No.,Message_body,Label,Case Normalization
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim..."


In [49]:
# Handling contractions

import contractions

def fix_contractions(data):
    expanded_corpus = [contractions.fix(text) for text in data]
    return expanded_corpus

df["Case Normalization"] = fix_contractions(df["Case Normalization"])
df.head(1)

,S. No.,Message_body,Label,Case Normalization
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim..."


In [50]:
# Spell checking

from textblob import TextBlob

# def correct_with_textblob(text):
#     blob = TextBlob(text)
#     corrected=blob.correct()
#     return str(corrected)

def correct_with_textblob_(data):
    corrected_data = [str(TextBlob(text).correct()) for text in data]
    return corrected_data

df["Case Normalization"] = correct_with_textblob_(df["Case Normalization"])
df.head(1)

,S. No.,Message_body,Label,Case Normalization
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim..."


b. Noise Removal

In [53]:
# Removing numbers/digits

def remove_numbers(text):
    result = re.sub(r'\d+', '', text) 
    return result

df['Noise Removal'] = [remove_numbers(text) for text in df["Case Normalization"]]
df.head(1)

,S. No.,Message_body,Label,Case Normalization,Noise Removal
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...","upgrdcentre orange customer, you may now claim..."


In [54]:
# Removing Punctuation & Special Characters

regular_punct = list(string.punctuation)

def remove_punctuation(text, punct_list):
    for punc in punct_list:
        if punc in text:
            text = text.replace(punc, '')
    return text.strip()

df["Noise Removal"] = [remove_punctuation(text, regular_punct) for text in df["Noise Removal"]]
df.head(1)

,S. No.,Message_body,Label,Case Normalization,Noise Removal
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...


In [55]:
# Handling double whitespace from text

def remove_whitespace(text):
    return " ".join(text.split())

df["Noise Removal"] = [remove_whitespace(text) for text in df["Noise Removal"]]
df.head(1)

,S. No.,Message_body,Label,Case Normalization,Noise Removal
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...


In [56]:
# Removal of URLs

url_pattern=[]
url_pattern.append(re.compile(r'https:?//\S+'))
url_pattern.append(re.compile(r'http:?//\S+'))
url_pattern.append(re.compile(r'www.\S+'))

def remove_urls(text):
    for pattern in url_pattern:
        text=pattern.sub('', text)
    return text

df["Noise Removal"] = [remove_urls(text) for text in df["Noise Removal"]]
df.head(1)

,S. No.,Message_body,Label,Case Normalization,Noise Removal
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...


c. Tokenization

In [57]:
nltk.download('punkt_tab')

df["Tokenization"] = [nltk.word_tokenize(text) for text in df["Noise Removal"]]
df.head(1)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\matthieu.catteyfaye\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package punkt_tab is already up-to-date!


,S. No.,Message_body,Label,Case Normalization,Noise Removal,Tokenization
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...,"[upgrdcentre, orange, customer, you, may, now,..."


d. Stopwords removal

In [58]:
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\matthieu.catteyfaye\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package stopwords is already up-to-date!


In [59]:
def remove_stopwords(liste):
    stop_words = set(stopwords.words("english"))

    filtered_text = [word for word in liste if word.lower() not in stop_words]

    return filtered_text

df["Stopwords"] = [remove_stopwords(liste) for liste in df["Tokenization"]]
df.head(1)

,S. No.,Message_body,Label,Case Normalization,Noise Removal,Tokenization,Stopwords
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...,"[upgrdcentre, orange, customer, you, may, now,...","[upgrdcentre, orange, customer, may, claim, fr..."


e-1. Stemming

In [60]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()

def stem_words(liste):
    stems = [stemmer.stem(word) for word in liste]
    return stems

df["Stemming"] = [stem_words(liste) for liste in df["Stopwords"]]
df.head(1)

,S. No.,Message_body,Label,Case Normalization,Noise Removal,Tokenization,Stopwords,Stemming
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...,"[upgrdcentre, orange, customer, you, may, now,...","[upgrdcentre, orange, customer, may, claim, fr...","[upgrdcentr, orang, custom, may, claim, free, ..."


e-2. Lemmatization

In [61]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemma_words(liste):
    lemmas = [lemmatizer.lemmatize(word) for word in liste]
    return lemmas

df["Lemmatization"] = [lemma_words(liste) for liste in df["Stopwords"]]
df.head(1)

,S. No.,Message_body,Label,Case Normalization,Noise Removal,Tokenization,Stopwords,Stemming,Lemmatization
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...,"[upgrdcentre, orange, customer, you, may, now,...","[upgrdcentre, orange, customer, may, claim, fr...","[upgrdcentr, orang, custom, may, claim, free, ...","[upgrdcentre, orange, customer, may, claim, fr..."


In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   S. No.              125 non-null    int64 
 1   Message_body        125 non-null    str   
 2   Label               125 non-null    str   
 3   Case Normalization  125 non-null    str   
 4   Noise Removal       125 non-null    str   
 5   Tokenization        125 non-null    object
 6   Stopwords           125 non-null    object
 7   Stemming            125 non-null    object
 8   Lemmatization       125 non-null    object
dtypes: int64(1), object(4), str(4)
memory usage: 8.9+ KB


In [63]:
df.to_csv("../data/SMS_test_clean.csv", index=False)